# Movement Classification
This notebook loads detections which are previously organized into observation windows and performs the movement classification on these observation windows. It organizes them together with the tracking results to enable evaluation with the PolyMot evaluation function.

File naming:
* Input detections matched for observation windows: "_matched_dets.json"
* Ordered detections: "data/tmp/detections_ordered_scenes.json"
* Detections with z-score values: "data/' + test_function.__name__ + 'min_' + str(min_len) + 'ess' + str(ess) + '.json'"
* Combined z-scores and tracking results for evaluation: "data/tracking_result_' + test_function.__name__ + 'min_' + str(min_len) + 'ess' + str(ess) + '.json'"

In [1]:
import os, json, sys
import argparse
sys.path.append('../')
from utils.io import load_file
from tqdm import tqdm
sys.path.append('/workspaces/Poly-MOT/nuscenes_devkit_uncertainty/python-sdk')
from nuscenes.nuscenes import NuScenes


### File loading and nuScenes devkit initialization

In [2]:
det_file = '/workspaces/Poly-MOT/result/jittering_edgarscenes/matched_dets.json'
first_token_file = '/workspaces/Poly-MOT/data/utils/first_token_table/trainval/edgarscenes_first_token.json'
dataset_path = '/workspaces/Poly-MOT/dataset/edgarscenes'
dataset_version = 'trainval'  # or 'test'

In [3]:
nusc = NuScenes(version='v1.0-' + dataset_version, dataroot=dataset_path,
                        verbose=True)
#frame_num = 6019 if dataset_version == 'trainval' else 6008

Loading NuScenes tables for version v1.0-trainval...
24 category,
8 attribute,
4 visibility,
5081 instance,
154 sensor,
154 calibrated_sensor,
98570 ego_pose,
22 log,
22 scene,
1960 sample,
118902 sample_data,
156228 sample_annotation,
22 map,
Done loading in 9.9 seconds.
Reverse indexing ...
Done reverse indexing in 0.3 seconds.


Saving the tokens

In [5]:
first_tokens = load_file(first_token_file)

# assert len(first_tokens) == 150

ordered_tokens = {}

for seq, first_token in enumerate(first_tokens):
    seq_tokens = []
    curr_token = first_token
    while curr_token != '':
        seq_tokens.append(curr_token)
        curr_token = nusc.get('sample', curr_token)['next']
    ordered_tokens[seq] = seq_tokens

with open('./data/edgarscenes_ordered_tokens.json', 'w') as f:
    json.dump(ordered_tokens, f)

Parsing /workspaces/Poly-MOT/data/utils/first_token_table/trainval/edgarscenes_first_token.json


It is also possible to load the tokens from the previously generated file in order to avoid nuScenes initialization (which can take a couple of minutes)

In [6]:
ordered_tokens = load_file('./data/edgarscenes_ordered_tokens.json')
inverted_tokens = {}
for scene_idx, tokens in ordered_tokens.items():
    for token in tokens:
        inverted_tokens[token] = str(scene_idx)

Parsing ./data/edgarscenes_ordered_tokens.json


In [7]:
dets = load_file(det_file)

Parsing /workspaces/Poly-MOT/result/jittering_edgarscenes/matched_dets.json


In [8]:
# Class mappings
BICYCLE = 0
MOTORCYCLE = 3.0
PEDESTRIAN = 4.0
CAR = 2.0
BUS = 1.0
TRAILER = 5.0
TRUCK = 6.0

Organize the detections and make them accessable with the corresponding tracking id

In [9]:
dets_all_scenes = {
    # scene: {
    #   tracking_id: det
    # }
}
for scene_number, frames in ordered_tokens.items():
    scene = {}
    for i, frame_token in enumerate(frames):
        for det in dets[frame_token]:
            if det['class_label'] not in [CAR, BUS, BICYCLE, TRUCK, PEDESTRIAN]:
                continue
            tra_id = det['tracking_id']
            if tra_id not in scene: 
                scene[tra_id] = {
                    'translation': [None] * i,
                    'velocity': [None] * i,
                    'trans_var': [None] * i,
                    'vel_var': [None] * i
                }
            
            scene[tra_id]['translation'].append(det['translation'])
            scene[tra_id]['velocity'].append(det['velocity'])
            scene[tra_id]['trans_var'].append(det['trans_var'])
            scene[tra_id]['vel_var'].append(det['vel_var'])
            
        # append None if det is missing for this frame
        for tra_id, det in scene.items():
            if len(det['translation']) == i:
                for key, value in det.items():
                    value.append(None)
    dets_all_scenes[scene_number] = scene

In [11]:
with open('./data/tmp_detections_ordered_scenes.json', 'w') as f:
    json.dump(dets_all_scenes, f)

## Movement Classifier

### Boolean as return

In [11]:
def perform_movement_z_test(observation_window, z_score_threshold=1.96):
    """
    Translated from C++ to Python. Performs a two-sample z-test on position data (x, y)
    and a one-sample z-test on velocity (vel_x, vel_y).

    Parameters:
        observation_window: List of observations, each with attributes:
            x_pos, y_pos,
            velocity_x, velocity_y,
            x_pos_uncertainty, y_pos_uncertainty,
            velocity_x_uncertainty, velocity_y_uncertainty
        z_score_threshold: Z-score threshold for movement detection.

    Returns:
        A list of four booleans [pos_x_significant, pos_y_significant, vel_x_stationary, vel_y_stationary].
        · pos_x_significant/pos_y_significant = True if position difference is above the threshold
        · vel_x_stationary/vel_y_stationary = True if velocity is above threshold
    """
    import math

    num_observations = len(observation_window)
    if num_observations == 0:
        return [False, False, False, False]

    # Split the observations roughly in half for the two-sample z-test on position
    n = int(math.ceil(num_observations / 2.0))
    m = num_observations - n

    # Mean and uncertainty values for the two-sample z-test (x, y)
    left_mean = [0.0, 0.0]
    right_mean = [0.0, 0.0]
    left_uncertainty = [0.0, 0.0]
    right_uncertainty = [0.0, 0.0]

    # Mean and uncertainty values for the one-sample z-test (vel_x, vel_y)
    velocity_mean = [0.0, 0.0]
    velocity_uncertainty = [0.0, 0.0]

    # (x_pos_test, y_pos_test, vel_x_test, vel_y_test)
    z_test_results = [False, False, False, False]

    # Accumulate sums for means and uncertainties
    for i, obs in enumerate(observation_window):
        if i < n:
            mean = left_mean
            unc = left_uncertainty
        else:
            mean = right_mean
            unc = right_uncertainty

        mean[0] += obs[0]
        mean[1] += obs[1]
        unc[0] += obs[4]
        unc[1] += obs[5]

        velocity_mean[0] += obs[2]
        velocity_mean[1] += obs[3]
        velocity_uncertainty[0] += obs[6]
        velocity_uncertainty[1] += obs[7]

    # Two-sample z-test for positions
    if n > 0 and m > 0:
        for idx in range(2):
            left_mean[idx] /= n
            right_mean[idx] /= m

            # Note: dividing by n*n or m*m is deriving the average of the variance across samples
            left_uncertainty[idx] /= float(n * n)
            right_uncertainty[idx] /= float(m * m)

            combined_variance = left_uncertainty[idx] + right_uncertainty[idx]
            # Avoid division by zero
            if combined_variance < 1e-15:
                # If variance is extremely small, treat it as zero
                z_test_results[idx] = False
            else:
                z_score = abs(left_mean[idx] - right_mean[idx]) / math.sqrt(combined_variance)
                z_test_results[idx] = (z_score > z_score_threshold)

    # One-sample z-test for velocities
    if num_observations > 0:
        for idx in range(2):
            velocity_mean[idx] /= float(num_observations)
            velocity_uncertainty[idx] /= float(num_observations * num_observations)

            if velocity_uncertainty[idx] < 1e-15:
                # If variance is extremely small, treat it as zero
                z_test_results[idx + 2] = False
            else:
                z_score = abs(velocity_mean[idx]) / math.sqrt(velocity_uncertainty[idx])
                # True here implies the velocity is within the stationary threshold
                z_test_results[idx + 2] = (z_score > z_score_threshold)

    return z_test_results

However, it is often more convenient to just get back the z-score and do the thresholding later.

### Z-score as return

In [12]:
def PRC_z_test_XY(observation_window, effective_sample_size):
    """
    Performs a two-sample z-test on position data (x, y) and returns the z_score values. 
    Meant for creating a PR curve to select the z_score threshold in the next step
    

    Parameters:
        observation_window: List of observations, each with attributes:
            x_pos, y_pos,
            velocity_x, velocity_y,
            x_pos_uncertainty, y_pos_uncertainty,
            velocity_x_uncertainty, velocity_y_uncertainty
       
    Returns:
        Z-score for the observation window mean shift
    """
    import math

    num_observations = len(observation_window)
    if num_observations == 0:
        return [False, False]

    # Split the observations roughly in half for the two-sample z-test on position
    n = int(math.ceil(num_observations / 2.0))
    m = num_observations - n

    # Mean and uncertainty values for the two-sample z-test (x, y)
    left_mean = [0.0, 0.0]
    right_mean = [0.0, 0.0]
    left_uncertainty = [0.0, 0.0]
    right_uncertainty = [0.0, 0.0]

    # (x_pos_test, y_pos_test)
    z_test_results = [False, False]

    # Accumulate sums for means and uncertainties
    for i, obs in enumerate(observation_window):
        if i < n:
            mean = left_mean
            unc = left_uncertainty
        else:
            mean = right_mean
            unc = right_uncertainty

        mean[0] += obs[0]
        mean[1] += obs[1]
        unc[0] += obs[4]
        unc[1] += obs[5]

    # Two-sample z-test for positions
    if n > 0 and m > 0:
        for idx in range(2):
            left_mean[idx] /= n
            right_mean[idx] /= m

            # Note: dividing by n*n or m*m is deriving the average of the variance across samples
            # Note: dividing only by n or m is using the observed variance mean with an effective sample size of 1 due to clustering
            left_uncertainty[idx] /= float(effective_sample_size * n)
            right_uncertainty[idx] /= float(effective_sample_size * m)

            combined_variance = left_uncertainty[idx] + right_uncertainty[idx]
            # Avoid division by zero
            if combined_variance < 1e-15:
                # If variance is extremely small, treat it as zero
                z_test_results[idx] = False
            else:
                z_score = abs(left_mean[idx] - right_mean[idx]) / math.sqrt(combined_variance)
                z_test_results[idx] = z_score 

    

    return max(z_test_results)

Run the movement classifier.

min_len: minimal observation window length

max_len = maximal observation window length

ess = effective sample sized (cluster the observations in each half of the window to somewhat account for the dependency of the observations)

In [13]:
test_function = PRC_z_test_XY
min_len = 2
max_len = 10
ess = 1
movement_result = {}
for scene, objects in dets_all_scenes.items():
    movement_result[scene] = {}
    for object, dets in objects.items():
        observation_window = [] # observation consists of x, y, vx, vy, var_x, var_y, var_vx, var_vy
        for t in range(len(dets['translation'])):
            frame_token = ordered_tokens[scene][t]
            if frame_token not in movement_result[scene]:
                movement_result[scene][frame_token] = {}
            if object not in movement_result[scene][frame_token]:
                movement_result[scene][frame_token][object] = None
            if dets['translation'][t] is not None:
                observation = dets['translation'][t][:2] + dets['velocity'][t][:2] + dets['trans_var'][t][:2] + dets['vel_var'][t][:2]
                observation_window.append(observation)
                
            if len(observation_window) > max_len:
                observation_window.pop(0)
            if len(observation_window) >= min_len:      
                test_result = test_function(observation_window, ess) 
                movement_result[scene][frame_token][object] = test_result




In [14]:

with open('./data/edgarscenes_' + test_function.__name__ + 'min_' + str(min_len) + 'ess' + str(ess) + '.json', 'w') as f:
    json.dump(movement_result, f)

### Append the movement classifier results to the tracking results

In [16]:
# Append motion_state to results.json
tracking_result = load_file('/workspaces/Poly-MOT/result/jittering_edgarscenes/tracking_results.json')
z_test_result = load_file('./data/edgarscenes_' + test_function.__name__ + 'min_' + str(min_len) + 'ess' + str(ess) + '.json')

Parsing /workspaces/Poly-MOT/result/jittering_edgarscenes/tracking_results.json
Parsing ./data/edgarscenes_PRC_z_test_XYmin_2ess1.json


In [17]:
for frame_token, object_list in tracking_result['results'].items():
    scene = inverted_tokens[frame_token]
    # for object in object_list:
    #     if object['tracking_id'] in z_test_result[scene][frame_token]:
    #         if z_test_result[scene][frame_token][object['tracking_id']]:
    #             state = 'moving'
    #         elif z_test_result[scene][frame_token][object['tracking_id']] == None:
    #             state = None
    #         else:
    #             state = 'static'
    #         object.update({'attribute_name': state})
    for object in object_list:
        if object['tracking_id'] in z_test_result[scene][frame_token]:
            if z_test_result[scene][frame_token][object['tracking_id']]:
                state = z_test_result[scene][frame_token][object['tracking_id']]
            elif z_test_result[scene][frame_token][object['tracking_id']] == None:
                 state = None
            object.update({'attribute_name': state})

In [18]:
with open('./data/edgarscenes_tracking_result_' + test_function.__name__ + 'min_' + str(min_len) + 'ess' + str(ess) + '.json', 'w') as f:
    json.dump(tracking_result, f)